# 1c. Build your own cache (optional)

Optional self-study: no other notebook depends on it.

Every other notebook in this course downloads caches that were built in advance. This one
builds a small cache itself, so you can see both halves of the pipeline separately, and
what each costs.

- **Creating a cache is the expensive half.** It runs the encoder over every image. In
  [01_the_cache.ipynb](01_the_cache.ipynb) §1.4, `dinov2-small` encoded about 300 images/s
  on the GPU (`mps`) of an Apple M5 Pro laptop and about 75 images/s on its CPU. Older
  laptops without a usable GPU are slower than that; the build cell below prints the rate
  on your machine, so you know your own number.
- **Loading a cache and training on it is the cheap half.** Loading is a disk read; training
  a head on a few hundred vectors takes well under a second, on any machine.

To keep the expensive half small, this notebook encodes `N_IMAGES` training images (500)
plus a quarter of that for testing: 625 images, a few seconds at the GPU rate above and
under ten seconds at the CPU rate. Divide 625 by your own rate for your machine.

In [ ]:
import json
import sys
import time
from pathlib import Path

import datasets
import torch
import transformers
from datasets import load_dataset
from huggingface_hub.utils import disable_progress_bars as hub_quiet
from huggingface_hub.utils import logging as hub_logging
from loguru import logger
from torch import nn
from vectormesh import ImageVectorizer, VectorCache

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
hub_quiet()
hub_logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

torch.manual_seed(0)

N_IMAGES = param(500, test=16)       # the expensive part: every image goes through the encoder
CACHE_DIR = Path("own_caches").resolve()

SOURCE = "tanganke/eurosat"          # swap this for E2: any HF image dataset with train/test splits
REVISION = "43750fc422c7fbd4289e2df3f7473617a5937acc"   # pin the source; None means "latest"

# the tag names the preprocessing: a different slice size is a different cache
TRAIN_TAG = f"mini_eurosat_train_{N_IMAGES}"
TEST_TAG = f"mini_eurosat_test_{N_IMAGES}"


## 1c.1 The input: a small slice of a source dataset

`scripts/build_cache.py`, which builds the published caches, follows one rule that matters
here too: **shuffle before selecting.** Several hub datasets are sorted by label (DTD is),
so a plain `.select(range(n))` returns a handful of classes instead of a smaller sample of
the whole task. `source_idx` is attached before the shuffle, so it keeps pointing at the
row's position in the *original* split: that is how a cache row finds its image again.

The input to the encoder is an ordinary Hugging Face `Dataset` with an `image` column of
PIL images.

In [ ]:
def prepare(split: str, n: int, seed: int = 0) -> datasets.Dataset:
    full = load_dataset(SOURCE, split=split, revision=REVISION)
    full = full.add_column("source_idx", list(range(len(full))))
    return full.shuffle(seed=seed).select(range(min(n, len(full))))


train_raw = prepare("train", N_IMAGES)
test_raw = prepare("test", max(N_IMAGES // 4, 8))

print(train_raw)
print("\nfeatures:", train_raw.features)
first = train_raw[0]
print(f"\nrow 0: image {first['image'].size} {first['image'].mode}, "
      f"label {first['label']} ({train_raw.features['label'].names[first['label']]}), "
      f"source_idx {first['source_idx']}")
print(f"classes in the train slice: {len(set(train_raw['label']))} of {train_raw.features['label'].num_classes}")
first["image"]

## 1c.2 Create the cache

`VectorCache.get_or_create` looks in `CACHE_DIR` for a cache with this `dataset_tag` and
this encoder. If it finds one, it loads it. If not, it calls `VectorCache.create`, which

1. runs the vectorizer over the `image` column, batch by batch,
2. drops the `image` column (`remove_columns`), so the cache holds vectors and labels only,
3. writes the result to a new folder in `CACHE_DIR`.

The cell reports which of the two happened and how long it took. The first run builds;
every later run loads.

In [ ]:
vectorizer = ImageVectorizer(model_name="facebook/dinov2-small")
print(f"encoder {vectorizer.model_name}: {vectorizer.get_hidden_size} numbers per image, device {vectorizer.device}\n")


def build_or_load(tag: str, dataset: datasets.Dataset) -> VectorCache:
    before = set(CACHE_DIR.glob("*")) if CACHE_DIR.exists() else set()
    start = time.time()
    cache = VectorCache.get_or_create(
        CACHE_DIR, dataset_tag=tag, vectorizer=vectorizer, dataset=dataset, remove_columns=["image"]
    )
    secs = time.time() - start
    if set(CACHE_DIR.glob("*")) - before:
        print(f"{tag}: BUILT {len(cache)} rows in {secs:.1f}s -> {len(cache) / secs:.0f} images/s")
    else:
        print(f"{tag}: LOADED {len(cache)} rows from disk in {1000 * secs:.0f} ms (no encoding)")
    return cache


train_cache = build_or_load(TRAIN_TAG, train_raw)
test_cache = build_or_load(TEST_TAG, test_raw)

## 1c.3 What `create` wrote to disk

One folder per split, named `<timestamp>_<dataset_tag>_<encoder>_<column>`. `get_or_create`
finds a cache by the last three parts, which is why the tag includes `N_IMAGES`: with a
different slice size, the old cache would otherwise be loaded as if it matched. Here is
what is inside the training folder:

In [ ]:
folder = sorted(CACHE_DIR.glob(f"*_{TRAIN_TAG}_dinov2-small_embed"))[-1]
print(f"{folder.name}/")
for f in sorted(folder.iterdir()):
    print(f"    {f.name:<28} {f.stat().st_size / 1024:8.1f} KB")

print("\nmetadata.json:")
print(json.dumps(json.loads((folder / "metadata.json").read_text()), indent=2))

| file | written by | holds |
|---|---|---|
| `data-00000-of-00001.arrow` | Hugging Face `datasets` | the rows: `label`, `source_idx`, `embed`. 500 × 384 float32 numbers are 750 KB, nearly the whole file |
| `dataset_info.json` | `datasets` | the schema: column types, and the class names of `label` |
| `state.json` | `datasets` | bookkeeping: which Arrow files make up the dataset, and its fingerprint |
| `metadata.json` | vectormesh | what produced the vectors: the encoder (`model_tag`), `hidden_size`, `tensordtype`, the columns, the row count, the vectormesh version and the creation time |

The `image` column is gone: the cache holds no pixels. To see an image again, look up
`source_idx` in the source dataset at the pinned `REVISION`.

`metadata.json` is the file you read in [01_the_cache.ipynb](01_the_cache.ipynb) §1.2 for a
published cache. A published cache is this same kind of folder, one per split, uploaded
to the hub by `scripts/publish_cache.py`.

## 1c.4 What `load` gives you

`VectorCache.load(folder)` reads that folder back. `from_hub` in the other notebooks does
the same after downloading the folder. Here is what you get:

In [ ]:
cache = VectorCache.load(folder)

print("object       :", type(cache).__name__, "-", cache.name)
print("rows         :", len(cache))
print("columns      :", cache.column_names)
print("vector cols  :", cache.vector_columns, f"(dim {cache.dim})")
print("classes      :", cache.num_classes)
print("features     :", cache.features)

print("\none row, cache[0] -- a dict of tensors, what a DataLoader hands out:")
for name, value in cache[0].items():
    print(f"    {name:<11} shape {tuple(value.shape)!s:<7} dtype {value.dtype}")

column = cache["embed"]
print(f"\ncache['embed']       -> {type(column).__name__}: a reference to the column on disk")
X = column[:]
print(f"cache['embed'][:]    -> {type(X).__name__} {tuple(X.shape)} {X.dtype}")
print(f"cache['embed'][:8]   -> {tuple(column[:8].shape)}")
print(f"cache['label'][:]    -> {tuple(cache['label'][:].shape)} {cache['label'][:].dtype}")

Two ways to get at the vectors:

- `cache["embed"]` is a **lazy column**. Nothing has been read yet; the data stays in the
  Arrow file on disk, which the operating system maps into memory on demand.
- Slicing it, `cache["embed"][:]` or `[:8]`, **materialises** those rows as one torch tensor
  of shape `(rows, 384)`. For caches of this size (a few thousand rows, a few MB), read
  everything at once and train on the tensor, as the other notebooks do. For a cache larger
  than your memory, index in batches instead.

`cache[i]` gives one row as a dict of tensors, which is what a `DataLoader` uses.

Because the folder now exists, asking for the same cache again is a load, not a build:

In [ ]:
train_cache = build_or_load(TRAIN_TAG, train_raw)

## 1c.5 The cheap half: use the cache

First a sanity check without training: the 1-nearest-neighbour classifier from
[01_the_cache.ipynb](01_the_cache.ipynb) §1.1. With only about 50 images per class it will
not match the full cache, but it should be far above chance. Then train a linear head and
time it.

In [ ]:
Xtr, ytr = train_cache["embed"][:], train_cache["label"][:]
Xte, yte = test_cache["embed"][:], test_cache["label"][:]

A = nn.functional.normalize(Xtr, dim=1)
B = nn.functional.normalize(Xte, dim=1)
knn = (ytr[(B @ A.T).argmax(dim=1)] == yte).float().mean()
print(f"1-NN, no training : {knn:.4f}   (chance {1 / train_cache.num_classes:.4f})")

torch.manual_seed(0)
model = nn.Linear(train_cache.dim, train_cache.num_classes)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
start = time.time()
for _ in range(50):
    for i in range(0, len(Xtr), 64):
        opt.zero_grad()
        nn.functional.cross_entropy(model(Xtr[i:i + 64]), ytr[i:i + 64]).backward()
        opt.step()
head_secs = time.time() - start
with torch.no_grad():
    head_acc = (model(Xte).argmax(1) == yte).float().mean()
print(f"linear head       : {head_acc:.4f}   trained 50 epochs on {len(Xtr)} rows in {head_secs:.2f}s")

Compare the two halves. Encoding the 625 images took seconds (§1c.2); on a CPU it takes
several times longer. Both classifiers on top took under a second together. 1-NN gets
about 0.86 from 50 images per class and the linear head about 0.90; on the full 6000-row
cache the same two methods give 0.90 and 0.94 ([01_the_cache.ipynb](01_the_cache.ipynb)
§1.5). Every further experiment on this cache pays only for the cheap half.

## When 500 images is not enough

This notebook shows the mechanism; it does not build a cache anyone should train on. For a
full build at the size of the published caches, use `scripts/build_cache.py`:

```bash
uv run python scripts/build_cache.py \
    --source tanganke/eurosat --encoder facebook/dinov2-small --tag eurosat
```

or `make build-lesson1` for every dataset and encoder this lesson uses. The time that takes
is the number of images divided by your rate from §1c.2: 8000 EuroSAT images at 75 images/s
is just under two minutes; at a tenth of that rate, about eighteen.

## Exercises

**E1 — trigger the mismatch guard.** Rerun the build cell with a *different* `vectorizer`
(say `microsoft/resnet-18`) but the *same* `dataset_tag` (`TRAIN_TAG`).
`get_or_create` builds a new folder, because the encoder is part of the folder name. Then
rename that new folder by hand so its encoder part reads `dinov2-small`, and call
`build_or_load` with the DINOv2 vectorizer again. `get_or_create` should raise rather than
hand you resnet vectors under a DINOv2 name. Read the error message: it names exactly what
disagreed.

**E2 — build a cache nobody has published.** Pick a different encoder (anything the hub's
`AutoModel` can load) or a different image dataset. An LLM with hub search is a fast way to
shortlist candidates, but the reason *why* this encoder suits this data should be yours.
Change `SOURCE` (and set `REVISION` to `None` or to a commit of the new dataset) and the
vectorizer, give it a new `dataset_tag`, and rerun. Check the metadata, then run the 1-NN
sanity check on it.